# NeuroGS-Codec: Topology-aware Rate–Distortion 3D Gaussian Volume Codec (Starter Notebook)

**Purpose:** End-to-end, ready-to-run baseline for compressing a **3D microscopy volume** (voxel intensities) using an anisotropic 3D Gaussian mixture with:
- neurite-biased **structure weighting** (tubularity proxy)
- **rate–distortion** training with quantization-in-the-loop (STE)
- optional **densify/prune** hooks (microscopy-aware)

**Created:** 2026-02-03 09:14:53

> This notebook is a clean research scaffold. It prioritizes clarity over maximum speed.


## 0) Setup

In [1]:
!pip install tifffile

In [2]:

# If running in Colab, uncomment:
# !pip -q install tifffile scipy tqdm

import os
import math
import time
from dataclasses import dataclass
from typing import Tuple, Dict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import tifffile as tiff
except ImportError as e:
    raise ImportError("Please install tifffile: pip install tifffile") from e

from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_float32_matmul_precision("high")
print("Device:", DEVICE)


Device: cuda


## 1) Load a 3D volume (.tif stack)

In [3]:

# ---- User config ----
TIF_PATH = "../../data/dataset/10-2900-control-cell-05_cropped_corrected.tif"   # <-- set this
# Example anisotropic spacing (microns). Replace with your microscope metadata.
VOXEL_SPACING = (0.126, 0.126, 1.0)  # (dx, dy, dz)
# ---------------------

assert os.path.exists(TIF_PATH), f"File not found: {TIF_PATH}"

V_np = tiff.imread(TIF_PATH)  # expected shape: (Z, Y, X)
assert V_np.ndim == 3, f"Expected 3D volume, got shape {V_np.shape}"

print("Loaded:", V_np.shape, V_np.dtype, "min/max:", float(V_np.min()), float(V_np.max()))

# Normalize to [0,1] float32
V = V_np.astype(np.float32)
V = (V - V.min()) / (V.max() - V.min() + 1e-8)
V_t = torch.from_numpy(V).to(DEVICE)  # (Z,Y,X)


Loaded: (100, 647, 813) uint8 min/max: 2.0 255.0


## 2) Build physical coordinate grid (anisotropy-aware)

In [4]:

def make_coord_grid_zyx(shape_zyx: Tuple[int,int,int], spacing_xyz: Tuple[float,float,float], device: str):
    # Returns normalized physical coordinates in [-1,1]^3 for each voxel index.
    # shape_zyx: (Z,Y,X), spacing_xyz: (dx,dy,dz)
    Z, Y, X = shape_zyx
    dx, dy, dz = spacing_xyz

    xs = torch.arange(X, device=device) * dx
    ys = torch.arange(Y, device=device) * dy
    zs = torch.arange(Z, device=device) * dz

    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()
    z0, z1 = zs.min(), zs.max()

    xn = (xs - x0) / (x1 - x0 + 1e-8) * 2 - 1
    yn = (ys - y0) / (y1 - y0 + 1e-8) * 2 - 1
    zn = (zs - z0) / (z1 - z0 + 1e-8) * 2 - 1

    zz, yy, xx = torch.meshgrid(zn, yn, xn, indexing="ij")
    coords = torch.stack([xx, yy, zz], dim=-1)  # (Z,Y,X,3) in (x,y,z)
    return coords

coords_grid = make_coord_grid_zyx(V_t.shape, VOXEL_SPACING, DEVICE)
print("coords_grid:", coords_grid.shape, coords_grid.dtype)


coords_grid: torch.Size([100, 647, 813, 3]) torch.float32


## 3) Neurite-likelihood map M(x): simple tubularity proxy
Label-free prior derived from the volume (DoG + gradient magnitude).

In [5]:

def gaussian_blur_3d(vol: torch.Tensor, sigma: float) -> torch.Tensor:
    # Separable 3D Gaussian blur using 1D kernels and conv3d.
    if sigma <= 0:
        return vol
    radius = int(3 * sigma + 0.5)
    x = torch.arange(-radius, radius+1, device=vol.device, dtype=vol.dtype)
    k = torch.exp(-(x**2)/(2*sigma**2))
    k = k / (k.sum() + 1e-8)

    v = vol.unsqueeze(0).unsqueeze(0)  # (1,1,Z,Y,X)

    kx = k.view(1,1,1,1,-1)
    v = F.conv3d(v, kx, padding=(0,0,radius))

    ky = k.view(1,1,1,-1,1)
    v = F.conv3d(v, ky, padding=(0,radius,0))

    kz = k.view(1,1,-1,1,1)
    v = F.conv3d(v, kz, padding=(radius,0,0))

    return v[0,0]

@torch.no_grad()
def make_neurite_map(vol_zyx: torch.Tensor) -> torch.Tensor:
    v1 = gaussian_blur_3d(vol_zyx, sigma=0.8)
    v2 = gaussian_blur_3d(vol_zyx, sigma=2.0)
    dog = (v1 - v2).abs()

    dz = F.pad(vol_zyx[1:] - vol_zyx[:-1], (0,0,0,0,0,1))
    dy = F.pad(vol_zyx[:,1:] - vol_zyx[:,:-1], (0,0,0,1,0,0))
    dx = F.pad(vol_zyx[:,:,1:] - vol_zyx[:,:,:-1], (0,1,0,0,0,0))
    gmag = torch.sqrt(dx*dx + dy*dy + dz*dz + 1e-8)

    m = dog + 0.5 * gmag
    m = (m - m.min()) / (m.max() - m.min() + 1e-8)
    return m.clamp(0,1)

M = make_neurite_map(V_t)
print("M:", M.shape, "min/max:", float(M.min()), float(M.max()), "mean:", float(M.mean()))


M: torch.Size([100, 647, 813]) min/max: 0.0 1.0 mean: 0.0345943421125412


## 4) Sampling strategy (uniform + neurite-biased)

In [6]:

@torch.no_grad()
def sample_points(coords_grid: torch.Tensor, V: torch.Tensor, M: torch.Tensor,
                  n_uniform: int, n_biased: int):
    Z,Y,X,_ = coords_grid.shape
    total = Z*Y*X

    idx_u = torch.randint(0, total, (n_uniform,), device=V.device)
    zu = idx_u // (Y*X)
    yu = (idx_u % (Y*X)) // X
    xu = idx_u % X

    # Handle large tensors: torch.multinomial has a 2^24 category limit
    flatM = M.reshape(-1)
    max_categories = 2**24 - 1
    if total > max_categories:
        # Pre-sample candidate indices, then use multinomial on the subset
        candidate_idx = torch.randint(0, total, (max_categories,), device=V.device)
        candidate_probs = flatM[candidate_idx]
        candidate_probs = candidate_probs / (candidate_probs.sum() + 1e-8)
        selected = torch.multinomial(candidate_probs, n_biased, replacement=True)
        idx_b = candidate_idx[selected]
    else:
        probs = flatM / (flatM.sum() + 1e-8)
        idx_b = torch.multinomial(probs, n_biased, replacement=True)
    
    zb = idx_b // (Y*X)
    yb = (idx_b % (Y*X)) // X
    xb = idx_b % X

    z = torch.cat([zu, zb], dim=0)
    y = torch.cat([yu, yb], dim=0)
    x = torch.cat([xu, xb], dim=0)

    pts = coords_grid[z,y,x]     # (N,3)
    tgt = V[z,y,x]               # (N,)
    mval = M[z,y,x]              # (N,)
    return pts, tgt, mval

pts, tgt, mval = sample_points(coords_grid, V_t, M, 1024, 1024)
print("Sampled:", pts.shape, tgt.shape, "m mean:", float(mval.mean()))

Sampled: torch.Size([2048, 3]) torch.Size([2048]) m mean: 0.05287308245897293


## 5) Gaussian mixture model (anisotropic, rotation via quaternion)

In [7]:

def quat_to_rotmat(q):
    """
    Convert quaternion (w, x, y, z) to rotation matrix.
    q: [N, 4]
    Returns: [N, 3, 3]
    """
    w, x, y, z = q.unbind(-1)
    
    # 3x3 Rotation matrix elements
    # Row 1
    r00 = 1 - 2*(y**2 + z**2)
    r01 = 2*(x*y - w*z)
    r02 = 2*(x*z + w*y)
    
    # Row 2
    r10 = 2*(x*y + w*z)
    r11 = 1 - 2*(x**2 + z**2)
    r12 = 2*(y*z - w*x)
    
    # Row 3
    r20 = 2*(x*z - w*y)
    r21 = 2*(y*z + w*x)
    r22 = 1 - 2*(x**2 + y**2)
    
    # Stack to [N, 3, 3]
    row1 = torch.stack([r00, r01, r02], dim=-1)
    row2 = torch.stack([r10, r11, r12], dim=-1)
    row3 = torch.stack([r20, r21, r22], dim=-1)
    
    return torch.stack([row1, row2, row3], dim=-2)


class GaussianMixtureVolume(nn.Module):
    def __init__(self, N, init_means=None, init_amp=None):
        super().__init__()
        self.N = N
        
        # Means (mu): [N, 3] in [0,1] normalized coords
        if init_means is not None:
            self.mu = nn.Parameter(init_means.clone())
        else:
            self.mu = nn.Parameter(torch.rand(N, 3))
            
        # Log scales (log_s): [N, 3]. Initialize small.
        # -4.0 corresponds to sigma approx 0.018 (small for 128^3 volume)
        self.log_s = nn.Parameter(torch.zeros(N, 3).fill_(-4.0))
        
        # Rotation (q): [N, 4] unit quaternions. Init as identity (1,0,0,0).
        self.q = nn.Parameter(torch.zeros(N, 4))
        self.q.data[:, 0] = 1.0
        
        # Opacity/Amplitude (a): [N]. Can be negative if we model dark spots? 
        # Usually positive for additive model.
        if init_amp is not None:
             self.a = nn.Parameter(init_amp.clone())
        else:
             self.a = nn.Parameter(torch.ones(N) * 0.1)
             
        # Background offset (b): [1]. Learnable background level.
        self.b = nn.Parameter(torch.zeros(1))

    def forward(self, coords):
        """
        Evaluate GMM at coords [B, 3].
        Returns density [B].
        """
        # Ensure quaternions are normalized
        q_norm = self.q / torch.norm(self.q, dim=-1, keepdim=True).clamp(min=1e-10)
        
        # Convert to rotation matrices R: [N, 3, 3]
        R = quat_to_rotmat(q_norm)
        
        # Scales: [N, 3]
        s = torch.exp(self.log_s)
        
        # Construct Covariance Matrices Sigma = R * S^2 * R^T
        # We need inverse covariance (Precision matrix) for evaluation:
        # Sigma^-1 = R * S^-2 * R^T
        # This is more numerically stable than inverting explicitly.
        
        s_inv_sq = 1.0 / (s**2 + 1e-10) # [N, 3]
        S_inv = torch.diag_embed(s_inv_sq) # [N, 3, 3]
        
        # Precision matrices P: [N, 3, 3]
        # P = R @ S_inv @ R.transpose(-2, -1)
        # But we can optimize calculation of (x-mu)^T @ P @ (x-mu)
        
        # Memory-efficient evaluation (Chunked if N*B is too large)
        # Here we assume N*B fits in memory (e.g. 5000 * 2000 = 10M floats = 40MB)
        
        # Expand dims for broadcasting: 
        # coords: [B, 1, 3]
        # mu:     [1, N, 3]
        x_minus_mu = coords.unsqueeze(1) - self.mu.unsqueeze(0) # [B, N, 3]
        
        # Rotate coordinates into Gaussian local frame
        # (x-mu) @ R ~ but we need R^T @ (x-mu) because P = R S^-2 R^T
        # v_local = (x-mu) @ R  -> [B, N, 3]
        # v_local^T @ S^-2 @ v_local  is the exponent
        
        # Note: R is [N, 3, 3]. We need to broadcast over B.
        # x_minus_mu is [B, N, 3].
        # We want [B, N, 3] vector-matrix multiply with [N, 3, 3]
        # Result [B, N, 3]
        
        # Einsum: b n i, n i j -> b n j
        v_local = torch.einsum('bni,nij->bnj', x_minus_mu, R)
        
        # Weighted sq norm in local frame
        # sum (v_local_k^2 * s_inv_sq_k)
        dist_sq = (v_local**2 * s_inv_sq.unsqueeze(0)).sum(dim=-1) # [B, N]
        
        # Gaussian weights: exp(-0.5 * dist_sq)
        weights = torch.exp(-0.5 * dist_sq) # [B, N]
        
        # Multiply by amplitude and sum
        density = (weights * self.a.unsqueeze(0)).sum(dim=-1) + self.b
        
        return density

@torch.no_grad()
def init_gaussians_from_neurite_map(coords_grid, V, M, N0: int):
    """
    Initialize Gaussian centers based on the importance map M.
    NOW WITH STRICT FILTERING: Only samples where V > 0.05.
    """
    Z,Y,X,_ = coords_grid.shape
    total = Z*Y*X
    flatM = M.reshape(-1)
    flatV = V.reshape(-1)
    
    # STRICT BACKGROUND FILTER
    # Zero out probability for any voxel below threshold
    # This prevents putting Gaussians effectively in the void
    valid_mask = (flatV > 0.05)
    flatM = flatM * valid_mask.float()
    
    # Check if we have valid candidates
    if valid_mask.sum() == 0:
        print("WARNING: No valid voxels found > 0.05! Initialization might happen in void.")
    
    # Handle large tensors: torch.multinomial has a 2^24 category limit
    max_categories = 2**24 - 1
    if total > max_categories:
        # Pre-sample candidate indices, then use multinomial on the subset
        candidate_idx = torch.randint(0, total, (max_categories,), device=M.device)
        candidate_probs = flatM[candidate_idx]
        candidate_probs = candidate_probs / (candidate_probs.sum() + 1e-8)
        selected = torch.multinomial(candidate_probs, N0, replacement=True)
        idx = candidate_idx[selected]
    else:
        probs = flatM / (flatM.sum() + 1e-8)
        idx = torch.multinomial(probs, N0, replacement=True)
    
    z = idx // (Y*X)
    y = (idx % (Y*X)) // X
    x = idx % X
    
    # Ensure coordinates are safely within bounds (though coords_grid should be correct)
    means = coords_grid[z,y,x].clamp(-1.0, 1.0)
    amp = V[z,y,x].clone()
    
    print(f"Init Stats: Mean range: [{means.min():.3f}, {means.max():.3f}]")
    
    return means, amp

# Clear any existing model/tensors from previous runs
import gc
gc.collect()
torch.cuda.empty_cache()

N0 = 5000  # increased to start with enough capacity
init_means, init_amp = init_gaussians_from_neurite_map(coords_grid, V_t, M, N0)
model = GaussianMixtureVolume(N0, init_means, init_amp).to(DEVICE)
# Use smaller initial scales for finer detail (-3.0 instead of -2.0)
model.log_s.data.fill_(-3.0)

print("Model N:", model.N)
print("Initialization Complete. Neurite priors applied.")

Init Stats: Mean range: [-1.000, 1.000]
Model N: 5000
Initialization Complete. Neurite priors applied.


## 6) Rate model (differentiable bitrate proxy)
Quantization-in-the-loop uses STE.

In [8]:

class LaplaceEntropyModel(nn.Module):
    def __init__(self, init_scale=1.0):
        super().__init__()
        self.log_b = nn.Parameter(torch.tensor(math.log(init_scale), device=DEVICE))

    def nll_bits(self, xq: torch.Tensor) -> torch.Tensor:
        b = torch.exp(self.log_b).clamp(1e-6, 1e3)
        logp = -math.log(2.0) - torch.log(b) - (xq.abs() / b)
        bits = (-logp / math.log(2.0)).mean()
        return bits

def ste_round(x: torch.Tensor) -> torch.Tensor:
    return (x.round() - x).detach() + x

@dataclass
class QuantSteps:
    mu: float = 1/2048
    log_s: float = 1/256
    q: float = 1/1024
    a: float = 1/1024
    b: float = 1/1024

H_mu   = LaplaceEntropyModel(init_scale=0.2).to(DEVICE)
H_logs = LaplaceEntropyModel(init_scale=0.5).to(DEVICE)
H_q    = LaplaceEntropyModel(init_scale=0.2).to(DEVICE)
H_a    = LaplaceEntropyModel(init_scale=0.5).to(DEVICE)
H_b    = LaplaceEntropyModel(init_scale=0.5).to(DEVICE)

Q = QuantSteps()


## 7) Losses: distortion + topology proxy + rate

In [9]:

def charbonnier(x, eps=1e-3):
    return torch.sqrt(x*x + eps*eps)

@torch.no_grad()
def extract_random_patch(vol: torch.Tensor, patch_zyx=(32,64,64)):
    Z,Y,X = vol.shape
    pz,py,px = patch_zyx
    z0 = torch.randint(0, max(1, Z-pz+1), (1,), device=vol.device).item()
    y0 = torch.randint(0, max(1, Y-py+1), (1,), device=vol.device).item()
    x0 = torch.randint(0, max(1, X-px+1), (1,), device=vol.device).item()
    return (z0,y0,x0), vol[z0:z0+pz, y0:y0+py, x0:x0+px]

def tv3d(p: torch.Tensor):
    dz = (p[1:] - p[:-1]).abs().mean()
    dy = (p[:,1:] - p[:,:-1]).abs().mean()
    dx = (p[:,:,1:] - p[:,:,:-1]).abs().mean()
    return dx + dy + dz

def patch_topology_loss(model: GaussianMixtureVolume, coords_grid: torch.Tensor, V: torch.Tensor,
                        patch_zyx=(16,32,32), tau=0.25, gamma=10.0):
    (z0,y0,x0), _ = extract_random_patch(V, patch_zyx)
    pz,py,px = patch_zyx
    coords = coords_grid[z0:z0+pz, y0:y0+py, x0:x0+px].reshape(-1,3)
    pred = model(coords).reshape(pz,py,px)

    P = torch.sigmoid(gamma*(pred - tau))
    tv = tv3d(P)

    P_ = P[None,None]
    low = F.avg_pool3d(P_, kernel_size=3, stride=1, padding=1)[0,0]
    hf = (P - low).abs().mean()
    return tv + 0.5*hf

# ========== New Regularizers ==========

def reconstruction_tv_loss(model: GaussianMixtureVolume, coords_grid: torch.Tensor, 
                           patch_zyx=(8,16,16)) -> torch.Tensor:
    """
    Direct TV regularization on raw reconstruction values.
    Encourages smooth intensity transitions while preserving edges.
    Uses a random patch for efficiency.
    """
    Z, Y, X, _ = coords_grid.shape
    pz, py, px = patch_zyx
    
    # Random patch location
    z0 = torch.randint(0, max(1, Z-pz+1), (1,), device=coords_grid.device).item()
    y0 = torch.randint(0, max(1, Y-py+1), (1,), device=coords_grid.device).item()
    x0 = torch.randint(0, max(1, X-px+1), (1,), device=coords_grid.device).item()
    
    coords = coords_grid[z0:z0+pz, y0:y0+py, x0:x0+px].reshape(-1, 3)
    pred = model(coords).reshape(pz, py, px)
    
    # Standard TV on raw reconstruction
    return tv3d(pred)

def ssim_loss_3d(pred: torch.Tensor, target: torch.Tensor, win=5, K1=0.01, K2=0.03) -> torch.Tensor:
    """
    Differentiable 3D SSIM loss (1 - SSIM) for structural similarity.
    Preserves local structure better than pixel-wise loss.
    """
    pred_ = pred[None, None]    # (1,1,Z,Y,X)
    tgt_ = target[None, None]
    pad = win // 2
    
    mu_p = F.avg_pool3d(pred_, win, stride=1, padding=pad)
    mu_t = F.avg_pool3d(tgt_, win, stride=1, padding=pad)
    
    sigma_p = F.avg_pool3d(pred_ * pred_, win, stride=1, padding=pad) - mu_p * mu_p
    sigma_t = F.avg_pool3d(tgt_ * tgt_, win, stride=1, padding=pad) - mu_t * mu_t
    sigma_pt = F.avg_pool3d(pred_ * tgt_, win, stride=1, padding=pad) - mu_p * mu_t
    
    C1, C2 = K1**2, K2**2
    ssim_map = ((2*mu_p*mu_t + C1) * (2*sigma_pt + C2)) / \
               ((mu_p**2 + mu_t**2 + C1) * (sigma_p + sigma_t + C2) + 1e-8)
    
    return 1.0 - ssim_map.mean()

def patch_ssim_loss(model: GaussianMixtureVolume, coords_grid: torch.Tensor, 
                    V: torch.Tensor, patch_zyx=(8,16,16)) -> torch.Tensor:
    """SSIM loss on a random patch for efficiency."""
    (z0,y0,x0), tgt_patch = extract_random_patch(V, patch_zyx)
    pz,py,px = patch_zyx
    coords = coords_grid[z0:z0+pz, y0:y0+py, x0:x0+px].reshape(-1,3)
    pred_patch = model(coords).reshape(pz,py,px)
    return ssim_loss_3d(pred_patch, tgt_patch, win=3)

def edge_aware_loss(model: GaussianMixtureVolume, coords_grid: torch.Tensor,
                    V: torch.Tensor, M: torch.Tensor, patch_zyx=(8,16,16)) -> torch.Tensor:
    """
    Extra loss weighted by gradient magnitude - focuses on edges/fine structures.
    """
    (z0,y0,x0), tgt_patch = extract_random_patch(V, patch_zyx)
    pz,py,px = patch_zyx
    coords = coords_grid[z0:z0+pz, y0:y0+py, x0:x0+px].reshape(-1,3)
    pred_patch = model(coords).reshape(pz,py,px)
    m_patch = M[z0:z0+pz, y0:y0+py, x0:x0+px]
    
    # Compute gradient magnitude of target as edge weight
    dz = F.pad(tgt_patch[1:] - tgt_patch[:-1], (0,0,0,0,0,1)).abs()
    dy = F.pad(tgt_patch[:,1:] - tgt_patch[:,:-1], (0,0,0,1,0,0)).abs()
    dx = F.pad(tgt_patch[:,:,1:] - tgt_patch[:,:,:-1], (0,1,0,0,0,0)).abs()
    edge_weight = (dz + dy + dx + m_patch).clamp(0, 1)
    
    # Weighted L1 loss focusing on edges
    diff = (pred_patch - tgt_patch).abs()
    return (edge_weight * diff).mean()

def sparsity_loss(model: GaussianMixtureVolume) -> torch.Tensor:
    """
    L1 sparsity on amplitudes to encourage fewer active Gaussians.
    Promotes compact representations by driving small amplitudes to zero.
    """
    return model.a.abs().mean()

def smoothness_loss(model: GaussianMixtureVolume) -> torch.Tensor:
    """
    Regularize Gaussian scales to encourage smooth, well-behaved Gaussians.
    Penalizes very small scales (sharp spikes) and extreme anisotropy.
    """
    s = torch.exp(model.log_s)  # (N, 3) actual scales
    
    # Penalize very small scales (prevents delta-like Gaussians)
    min_scale_penalty = torch.relu(0.01 - s).mean()
    
    # Penalize extreme anisotropy (ratio between max and min scale per Gaussian)
    s_max = s.max(dim=-1).values
    s_min = s.min(dim=-1).values + 1e-8
    anisotropy = (s_max / s_min - 1.0).clamp(min=0).mean()  # 0 if isotropic
    
    # Penalize very large scales (prevents overly blurry Gaussians)
    max_scale_penalty = torch.relu(s - 1.0).mean()
    
    return min_scale_penalty + 0.1 * anisotropy + 0.1 * max_scale_penalty

def overlap_loss(model: GaussianMixtureVolume, n_samples: int = 512) -> torch.Tensor:
    """
    Penalize overlapping Gaussians by sampling pairs and measuring their proximity.
    Encourages spatial separation and diverse coverage.
    """
    N = model.N
    if N < 2:
        return torch.tensor(0.0, device=model.mu.device)
    
    # Sample random pairs of Gaussians
    n_pairs = min(n_samples, N * (N - 1) // 2)
    idx_i = torch.randint(0, N, (n_pairs,), device=model.mu.device)
    idx_j = torch.randint(0, N, (n_pairs,), device=model.mu.device)
    
    # Avoid self-pairs
    mask = idx_i != idx_j
    idx_i, idx_j = idx_i[mask], idx_j[mask]
    
    if len(idx_i) == 0:
        return torch.tensor(0.0, device=model.mu.device)
    
    # Get positions and scales
    mu_i = model.mu[idx_i]  # (n_pairs, 3)
    mu_j = model.mu[idx_j]
    s_i = torch.exp(model.log_s[idx_i])  # (n_pairs, 3)
    s_j = torch.exp(model.log_s[idx_j])
    
    # Compute distance between centers
    dist = torch.norm(mu_i - mu_j, dim=-1)  # (n_pairs,)
    
    # Effective radius is the geometric mean of scales
    r_i = s_i.prod(dim=-1).pow(1/3)
    r_j = s_j.prod(dim=-1).pow(1/3)
    
    # Overlap occurs when distance < sum of effective radii
    # Soft penalty: exp(-dist / (r_i + r_j + eps))
    overlap = torch.exp(-dist / (r_i + r_j + 1e-4))
    
    return overlap.mean()


## 8) Training loop (RD + topology)

In [10]:
# Clear GPU memory from previous runs
import gc
import os
import math
import time
from typing import Tuple
import torch
import torch.nn as nn
from tqdm import tqdm

# Fix OOM fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gc.collect()
torch.cuda.empty_cache()

# ============================================================================
# Core Utilities (Redefined for Stability)
# ============================================================================

def safe_normalize(x: torch.Tensor, eps: float = 1e-20) -> torch.Tensor:
    """Safe normalization to prevent NaNs with zero-length vectors."""
    return x / torch.sqrt(torch.clamp(torch.sum(x * x, dim=-1, keepdim=True), min=eps))

# ============================================================================
# Adaptive Densification Controller (inspired by 3D Gaussian Splatting)
# ============================================================================

class DensificationController:
    """
    Gradient-based adaptive densification and pruning for GaussianMixtureVolume.
    Based on the approach from 3D Gaussian Splatting (gaussian_model.py).
    """
    def __init__(self, model: GaussianMixtureVolume, percent_dense: float = 0.01):
        self.model = model
        self.percent_dense = percent_dense
        
        # Gradient accumulation for densification decisions
        self.xyz_gradient_accum = torch.zeros((model.N, 1), device=DEVICE)
        self.denom = torch.zeros((model.N, 1), device=DEVICE)
        
    def reset_stats(self):
        """Reset gradient accumulation after densification."""
        N = self.model.N
        self.xyz_gradient_accum = torch.zeros((N, 1), device=DEVICE)
        self.denom = torch.zeros((N, 1), device=DEVICE)
    
    def add_densification_stats(self, grad_mu: torch.Tensor):
        """Accumulate gradient statistics for densification decisions."""
        if grad_mu is not None:
            grad_norm = grad_mu.norm(dim=-1, keepdim=True)
            self.xyz_gradient_accum += grad_norm
            self.denom += 1
    
    def get_scaling(self) -> torch.Tensor:
        """Get the scaling (sigma) of each Gaussian."""
        return torch.exp(self.model.log_s).clamp(1e-4, 10.0)
    
    @torch.no_grad()
    def densify_and_clone(self, grads: torch.Tensor, grad_threshold: float, scene_extent: float, max_entries: int = None):
        """Clone Gaussians with high gradient but small scale."""
        scaling = self.get_scaling()
        selected_pts_mask = (grads.squeeze() >= grad_threshold)
        selected_pts_mask = selected_pts_mask & (scaling.max(dim=1).values <= self.percent_dense * scene_extent)
        
        num_selected = int(selected_pts_mask.sum().item())
        if num_selected == 0:
            return 0
            
        # Limit number of clones
        if max_entries is not None and num_selected > max_entries:
            indices = torch.nonzero(selected_pts_mask).view(-1)
            if indices.numel() > 0:
                candidates_grads = grads.squeeze()[indices]
                _, topk = torch.topk(candidates_grads, k=max_entries)
                selected_pts_mask[:] = False
                selected_pts_mask[indices[topk]] = True
        
        new_mu = self.model.mu.data[selected_pts_mask].clone()
        new_log_s = self.model.log_s.data[selected_pts_mask].clone()
        new_q = self.model.q.data[selected_pts_mask].clone()
        new_a = self.model.a.data[selected_pts_mask].clone()
        
        self._cat_tensors(new_mu, new_log_s, new_q, new_a)
        return int(selected_pts_mask.sum().item())
    
    @torch.no_grad()
    def densify_and_split(self, grads: torch.Tensor, grad_threshold: float, scene_extent: float, N_split: int = 2, max_entries: int = None):
        """Split Gaussians with high gradient AND large scale."""
        scaling = self.get_scaling()
        n_init = self.model.N
        
        padded_grad = torch.zeros((n_init,), device=DEVICE)
        padded_grad[:grads.shape[0]] = grads.squeeze()
        
        selected_pts_mask = (padded_grad >= grad_threshold)
        selected_pts_mask = selected_pts_mask & (scaling.max(dim=1).values > self.percent_dense * scene_extent)
        
        num_selected = int(selected_pts_mask.sum().item())
        if num_selected == 0:
            return 0
            
        # Limit number of splits
        if max_entries is not None and num_selected > max_entries:
            indices = torch.nonzero(selected_pts_mask).view(-1)
            if indices.numel() > 0:
                candidates_grads = padded_grad[indices]
                _, topk = torch.topk(candidates_grads, k=max_entries)
                selected_pts_mask[:] = False
                selected_pts_mask[indices[topk]] = True
                num_selected = max_entries
        
        # Extract selected parameters
        scales_sel = scaling[selected_pts_mask]
        means_sel = self.model.mu.data[selected_pts_mask]
        qs_sel = self.model.q.data[selected_pts_mask]
        dim_sel = means_sel.shape[0]

        # Prepare output tensors
        new_mu_list = []
        new_log_s_list = []
        new_q_list = []
        new_a_list = []

        # ====================================================================
        # CHUNKED PROCESSING to avoid OOM with large BMM
        # ====================================================================
        chunk_size = 500  # Process 500 points at a time
        
        for i in range(0, dim_sel, chunk_size):
            end = min(i + chunk_size, dim_sel)
            
            # Get chunk
            s_chunk = scales_sel[i:end] # (B, 3)
            m_chunk = means_sel[i:end]  # (B, 3)
            q_chunk = qs_sel[i:end]     # (B, 4)
            B = end - i
            
            # Generate samples
            stds = s_chunk.repeat(N_split, 1) # (B*N_split, 3)
            means = torch.zeros((B*N_split, 3), device=DEVICE)
            samples = torch.normal(mean=means, std=stds) # (B*N_split, 3)
            
            # Rotate samples
            qn = safe_normalize(q_chunk)
            R = quat_to_rotmat(qn) # (B, 3, 3)
            R_rep = R.repeat(N_split, 1, 1) # (B*N_split, 3, 3)
            
            # BMM: (B*N_split, 3, 3) @ (B*N_split, 3, 1) -> (B*N_split, 3)
            rot_samples = torch.bmm(R_rep, samples.unsqueeze(-1)).squeeze(-1)
            
            # Add to means
            new_mu_chunk = rot_samples + m_chunk.repeat(N_split, 1)
            new_mu_chunk = new_mu_chunk.clamp(-1, 1)
            
            # Prepare other attributes
            # Neural Split: Divide scale by small factor (1.1) to keep shape
            split_scaling_factor = 1.1
            new_log_s_chunk = self.model.log_s.data[selected_pts_mask][i:end].repeat(N_split, 1) - math.log(split_scaling_factor)
            new_q_chunk = self.model.q.data[selected_pts_mask][i:end].repeat(N_split, 1)
            new_a_chunk = self.model.a.data[selected_pts_mask][i:end].repeat(N_split).reshape(-1) / N_split
            
            new_mu_list.append(new_mu_chunk)
            new_log_s_list.append(new_log_s_chunk)
            new_q_list.append(new_q_chunk)
            new_a_list.append(new_a_chunk)
            
            # Free memory
            del R, R_rep, samples, rot_samples
        
        # Concatenate results
        new_mu = torch.cat(new_mu_list, dim=0)
        new_log_s = torch.cat(new_log_s_list, dim=0)
        new_q = torch.cat(new_q_list, dim=0)
        new_a = torch.cat(new_a_list, dim=0)
        
        self._cat_tensors(new_mu, new_log_s, new_q, new_a)
        
        prune_mask = torch.cat([selected_pts_mask, 
                                torch.zeros(N_split * num_selected, device=DEVICE, dtype=torch.bool)])
        self.prune_points(prune_mask)
        
        return num_selected * (N_split - 1)
    
    def _cat_tensors(self, new_mu, new_log_s, new_q, new_a):
        """Concatenate new Gaussians to the model."""
        add_count = new_mu.shape[0]
        
        self.model.mu = nn.Parameter(torch.cat([self.model.mu.data, new_mu], dim=0))
        self.model.log_s = nn.Parameter(torch.cat([self.model.log_s.data, new_log_s], dim=0))
        self.model.q = nn.Parameter(torch.cat([self.model.q.data, new_q], dim=0))
        new_a_flat = new_a.reshape(-1) if new_a.dim() > 1 else new_a
        self.model.a = nn.Parameter(torch.cat([self.model.a.data, new_a_flat], dim=0))
        
        self.model.N += add_count
        
        self.xyz_gradient_accum = torch.cat([self.xyz_gradient_accum, 
                                              torch.zeros((add_count, 1), device=DEVICE)], dim=0)
        self.denom = torch.cat([self.denom, torch.zeros((add_count, 1), device=DEVICE)], dim=0)
    
    @torch.no_grad()
    def prune_points(self, mask: torch.Tensor):
        """Remove Gaussians where mask is True."""
        valid_mask = ~mask
        if valid_mask.sum() == valid_mask.numel():
            return 0
        
        pruned = int((~valid_mask).sum().item())
        
        self.model.mu = nn.Parameter(self.model.mu.data[valid_mask])
        self.model.log_s = nn.Parameter(self.model.log_s.data[valid_mask])
        self.model.q = nn.Parameter(self.model.q.data[valid_mask])
        self.model.a = nn.Parameter(self.model.a.data[valid_mask])
        self.model.N = self.model.mu.shape[0]
        
        self.xyz_gradient_accum = self.xyz_gradient_accum[valid_mask]
        self.denom = self.denom[valid_mask]
        
        return pruned
    
    @torch.no_grad()
    def prune_low_amplitude(self, min_amplitude: float = 0.002):
        """Prune Gaussians with very low amplitude."""
        prune_mask = self.model.a.data.abs() < min_amplitude
        return self.prune_points(prune_mask)
    
    @torch.no_grad()  
    def prune_large_scale(self, max_scale: float = 0.5, scene_extent: float = 2.0):
        """Prune Gaussians that are too large."""
        scaling = self.get_scaling()
        prune_mask = scaling.max(dim=1).values > max_scale * scene_extent
        return self.prune_points(prune_mask)
    
    @torch.no_grad()
    def densify_and_prune(self, grad_threshold=0.0002, min_amplitude=0.002, 
                          scene_extent=2.0, max_scale=0.1, max_gaussians=None):
        """Full densification and pruning step."""
        grads = self.xyz_gradient_accum / (self.denom + 1e-8)
        grads[grads.isnan()] = 0.0
        
        # Calculate budget
        budget_clone = None
        budget_split = None
        if max_gaussians is not None:
             remaining = max_gaussians - self.model.N
             if remaining <= 0: remaining = 0
             # Assign half budget to each strategy
             budget_clone = remaining // 2
             budget_split = remaining - budget_clone 

        n_cloned = self.densify_and_clone(grads, grad_threshold, scene_extent, max_entries=budget_clone)
        n_split = self.densify_and_split(grads, grad_threshold, scene_extent, max_entries=budget_split)
        n_pruned_amp = self.prune_low_amplitude(min_amplitude)
        n_pruned_scale = self.prune_large_scale(max_scale, scene_extent)
        
        self.reset_stats()
        # Ensure temporary tensors are cleared
        torch.cuda.empty_cache()
        
        return {"cloned": n_cloned, "split": n_split, 
                "pruned_amp": n_pruned_amp, "pruned_scale": n_pruned_scale,
                "total_gaussians": self.model.N}


# ============================================================================
# NEURON-SPECIFIC LOSS FUNCTIONS
# ============================================================================
# Total Loss: L_Total = λ1*L_Voxel + λ2*L_Anisotropy + λ3*L_Sparsity + λ4*L_TV

def weighted_voxel_loss(pred: torch.Tensor, target: torch.Tensor, 
                        intensity_weight: float = 5.0) -> torch.Tensor:
    """
    Weighted Voxel Loss (L_Voxel) with intensity weighting.
    """
    w = 1 + intensity_weight * target
    return (w * (pred - target) ** 2).mean()


def anisotropy_loss(model: GaussianMixtureVolume, target_ratio: float = 0.1) -> torch.Tensor:
    """
    Anisotropy Loss (L_Anisotropy) — Crucial for Neurons.
    Refined: Uses linear scales (s) instead of variance (s^2) for numerical stability.
    """
    # Get scales (sigma) - Linear dimension
    s = torch.exp(model.log_s).clamp(1e-4, 10.0)  # (N, 3)
    
    # Sort scales: s_max >= s_med >= s_min
    s_sorted, _ = s.sort(dim=-1, descending=True)  # (N, 3)
    s_max = s_sorted[:, 0]
    s_med = s_sorted[:, 1]
    s_min = s_sorted[:, 2]
    
    # Ratio using linear scales (smoother gradients for thin structures)
    ratio = (s_med + s_min) / (s_max + 1e-8)
    
    # Penalize ratio above target (encourage elongation)
    loss = torch.relu(ratio - target_ratio).mean()
    
    return loss


def sparsity_loss_l1(model: GaussianMixtureVolume) -> torch.Tensor:
    """
    Sparsity & Opacity Loss (L_Sparsity).
    """
    return model.a.abs().mean()


def tv_loss_volume(pred_volume: torch.Tensor) -> torch.Tensor:
    """
    Total Variation / Continuity Loss (L_TV) on reconstructed volume.
    """
    # Compute gradients along each axis
    dz = pred_volume[1:, :, :] - pred_volume[:-1, :, :]  # Z gradient
    dy = pred_volume[:, 1:, :] - pred_volume[:, :-1, :]  # Y gradient  
    dx = pred_volume[:, :, 1:] - pred_volume[:, :, :-1]  # X gradient
    
    # L1 TV (sum of absolute gradients)
    tv = dz.abs().mean() + dy.abs().mean() + dx.abs().mean()
    
    return tv


def tv_loss_patch(model: GaussianMixtureVolume, coords_grid: torch.Tensor, 
                  patch_zyx: Tuple[int,int,int] = (8, 16, 16)) -> torch.Tensor:
    """
    TV loss computed on a random patch of the reconstructed volume.
    """
    Z, Y, X, _ = coords_grid.shape
    pz, py, px = patch_zyx
    
    # Random patch origin
    z0 = torch.randint(0, max(1, Z - pz), (1,)).item()
    y0 = torch.randint(0, max(1, Y - py), (1,)).item()
    x0 = torch.randint(0, max(1, X - px), (1,)).item()
    
    # Extract patch coordinates
    patch_coords = coords_grid[z0:z0+pz, y0:y0+py, x0:x0+px].reshape(-1, 3)
    
    # Reconstruct patch
    pred_patch = model(patch_coords).reshape(pz, py, px)
    
    return tv_loss_volume(pred_patch)


# ============================================================================
# Rate Computation (Rate-Distortion)
# ============================================================================

def compute_rate_bits(model: GaussianMixtureVolume) -> torch.Tensor:
    """Compute estimate of rate in bits."""
    mu_q   = ste_round(model.mu / Q.mu)
    logs_q = ste_round(model.log_s / Q.log_s)
    q_q    = ste_round(model.q / Q.q)
    a_q    = ste_round(model.a / Q.a)
    b_q    = ste_round(model.b / Q.b)

    bits = (H_mu.nll_bits(mu_q) +
            H_logs.nll_bits(logs_q) +
            H_q.nll_bits(q_q) +
            H_a.nll_bits(a_q) +
            H_b.nll_bits(b_q))
    return bits


# ============================================================================
# Training with Neuron-Specific Losses + Rate Distortion (Warm-up)
# ============================================================================
# L_Total = λ1*L_Voxel + λ2*L_Anisotropy + λ3*L_Sparsity + λ4*L_TV + λRate*L_Rate

def train(model: GaussianMixtureVolume,
          V: torch.Tensor, coords_grid: torch.Tensor, M: torch.Tensor,
          steps=3000, batch=2_000,
          # Loss weights
          lambda_voxel=1.0,          # λ1: Weighted voxel loss
          lambda_aniso=0.1,          # λ2: Anisotropy loss
          lambda_sparse=0.01,        # λ3: Sparsity loss (L1)
          lambda_tv=0.01,            # λ4: TV loss
          # Rate schedule parameters (Replaces constant lambda_rate)
          rate_warmup_steps=2000,
          rate_prune_steps=5000,
          rate_beta_soft=1e-7,
          rate_beta_refine=1e-6,
          # Additional parameters
          intensity_weight=5.0,      
          aniso_target=0.3,          
          tv_every=5,                
          tv_patch_size=(8, 16, 16), 
          lr=5e-3, lr_final=None,
          densify_enabled=True,
          densify_from_iter=500,
          densify_until_iter=8000,
          densify_every=200,
          grad_threshold=0.0005,
          min_amplitude=0.002,
          max_gaussians=20000):
    """
    Training with Rate-Distortion Warm-up Schedule.
    """
    controller = DensificationController(model, percent_dense=0.01) if densify_enabled else None
    
    # Configure parameter groups
    params = [
        {'params': [model.mu], 'lr': lr},
        {'params': [model.log_s], 'lr': lr * 2.0},  # Boost scale LR
        {'params': [model.q], 'lr': lr},
        {'params': [model.a], 'lr': lr},
        {'params': [model.b], 'lr': lr},
        {'params': list(H_mu.parameters()) + list(H_logs.parameters()) + 
                   list(H_q.parameters()) + list(H_a.parameters()) + list(H_b.parameters()), 'lr': lr}
    ]
    opt = torch.optim.Adam(params, lr=lr)

    losses = {"Voxel": [], "Aniso": [], "Sparse": [], "TV": [], "Rate": [], "Total": [], "N": []}
    densify_log = []
    t0 = time.time()
    
    last_tv = 0.0
    last_rate = 0.0
    
    if lr_final is None:
        lr_final = lr

    print("=" * 70)
    print("NEURON-SPECIFIC TRAINING (w/ RD WARM-UP SCHEDULE)")
    print("=" * 70)
    print(f"L_Total = λ1*L_Voxel + λ2*L_Aniso + λ3*L_Sparse + λ4*L_TV + β(t)*L_Rate")
    print(f"  λ1 (voxel)     = {lambda_voxel:.4f})")
    print(f"  λ2 (anisotropy)= {lambda_aniso:.4f}")
    print(f"  λ3 (sparsity)  = {lambda_sparse:.4f}")
    print(f"  λ4 (TV)        = {lambda_tv:.4f}")
    print(f"  Rate Schedule (β):")
    print(f"    0 - {rate_warmup_steps}: 0.0 (Discovery)")
    print(f"    {rate_warmup_steps} - {rate_prune_steps}: {rate_beta_soft:.1e} (Soft Pruning)")
    print(f"    {rate_prune_steps}+ : {rate_beta_refine:.1e} (Refinement)")
    print("=" * 70)

    for it in tqdm(range(steps), desc="Training"):
        # Rate Schedule Logic
        if it < rate_warmup_steps:
            current_lambda_rate = 0.0
        elif it < rate_prune_steps:
            current_lambda_rate = rate_beta_soft
        else:
            current_lambda_rate = rate_beta_refine
            
        # Learning rate decay
        current_lr = lr - (lr - lr_final) * (it / max(steps - 1, 1))
        for pg in opt.param_groups:
            if pg['params'][0] is model.log_s:
                 pg['lr'] = current_lr * 2.0
            else:
                 pg['lr'] = current_lr

        # Sample points
        n_u = batch // 2
        n_b = batch - n_u
        pts, tgt, mval = sample_points(coords_grid, V, M, n_u, n_b)

        # Forward pass
        pred = model(pts)
        
        # Losses
        L_voxel = weighted_voxel_loss(pred, tgt, intensity_weight=intensity_weight)
        
        L_aniso = anisotropy_loss(model, target_ratio=aniso_target) if lambda_aniso > 0 else torch.zeros(1, device=V.device)
        
        L_sparse = sparsity_loss_l1(model) if lambda_sparse > 0 else torch.zeros(1, device=V.device)
        
        # TV Loss (periodic)
        if lambda_tv > 0 and (it % tv_every) == 0:
            L_tv = tv_loss_patch(model, coords_grid, patch_zyx=tv_patch_size)
            last_tv = float(L_tv.detach().cpu())
        else:
            L_tv = torch.zeros(1, device=V.device)
            
        # Rate Loss (RD term)
        if current_lambda_rate > 0:
            # Note: we use sum() because the beta (1e-7) is scaled for TOTAL bits (approx 1,000,000)
            rate_bits = compute_rate_bits(model).sum() # Sum of bits across all Gaussians
            L_rate = rate_bits
            last_rate = float(rate_bits.detach().cpu())
        else:
            L_rate = torch.zeros(1, device=V.device)
            last_rate = 0.0

        # Total Loss
        current_L_tv = L_tv if lambda_tv > 0 and (it % tv_every) == 0 else torch.tensor(0.0, device=V.device)
            
        total = (lambda_voxel * L_voxel + 
                 lambda_aniso * L_aniso + 
                 lambda_sparse * L_sparse + 
                 lambda_tv * current_L_tv + 
                 current_lambda_rate * L_rate)

        # Backward
        opt.zero_grad(set_to_none=True)
        total.backward()
        
        # Accumulate grad stats
        if densify_enabled and controller is not None and model.mu.grad is not None:
            controller.add_densification_stats(model.mu.grad)
        
        opt.step()

        # Logging
        losses["Voxel"].append(float(L_voxel.detach().cpu()))
        losses["Aniso"].append(float(L_aniso.detach().cpu()))
        losses["Sparse"].append(float(L_sparse.detach().cpu()))
        losses["TV"].append(last_tv)
        losses["Rate"].append(last_rate)
        losses["Total"].append(float(total.detach().cpu()))
        losses["N"].append(model.N)

        # Densification
        if densify_enabled and controller is not None:
            if (densify_from_iter <= it < densify_until_iter) and ((it + 1) % densify_every == 0):
                # Dynamically relax max_scale during warm-up
                current_max_scale = 0.1 if it < 2000 else 0.05
                
                if model.N < max_gaussians:
                    stats = controller.densify_and_prune(
                        grad_threshold=grad_threshold,
                        min_amplitude=min_amplitude,
                        scene_extent=2.0,
                        max_scale=current_max_scale,
                        max_gaussians=max_gaussians
                    )
                    densify_log.append((it, stats))
                    
                    # Memory Cleanup & Optimizer Reset
                    del opt
                    gc.collect()
                    torch.cuda.empty_cache()
                    
                    # Update Entropy Models (if needed)
                    if hasattr(H_mu, 'update'): H_mu.update()
                    if hasattr(H_logs, 'update'): H_logs.update()
                    if hasattr(H_q, 'update'): H_q.update()
                    if hasattr(H_a, 'update'): H_a.update()
                    if hasattr(H_b, 'update'): H_b.update()
                    
                    params = [
                        {'params': [model.mu], 'lr': current_lr},
                        {'params': [model.log_s], 'lr': current_lr * 2.0},
                        {'params': [model.q], 'lr': current_lr},
                        {'params': [model.a], 'lr': current_lr},
                        {'params': [model.b], 'lr': current_lr},
                        {'params': list(H_mu.parameters()) + list(H_logs.parameters()) + 
                                   list(H_q.parameters()) + list(H_a.parameters()) + list(H_b.parameters()), 'lr': current_lr}
                    ]
                    opt = torch.optim.Adam(params, lr=current_lr)
                    
                    print(f"  [Densify @{it+1}] N={stats['total_gaussians']} (+{stats['cloned']} +{stats['split']} -{stats['pruned_amp']} -{stats['pruned_scale']})")

        # Progress
        if (it+1) % 200 == 0:
            dt = time.time() - t0
            print(f"iter {it+1:5d} | Vox={losses['Voxel'][-1]:.4f} Rate={last_rate:.1f}b (β={current_lambda_rate:.1e}) "
                  f"Tot={losses['Total'][-1]:.4f} N={model.N} | {dt:.1f}s")
            
            # Checkpoint
            if (it+1) % 1000 == 0:
                os.makedirs("checkpoint", exist_ok=True)
                torch.save(model.state_dict(), f"checkpoint/iter{it+1}.pt")
        
        if (it+1) % 500 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    return losses, densify_log


# ============================================================================
# Initialize and Train
# ============================================================================
gc.collect()
torch.cuda.empty_cache()

init_means, init_amp = init_gaussians_from_neurite_map(coords_grid, V_t, M, N0)
model = GaussianMixtureVolume(N0, init_means=init_means, init_amp=init_amp).to(DEVICE)
print(f"Model initialized with N={model.N} Gaussians")

# Train with Rate-Distortion Schedule
losses, densify_log = train(
    model, V_t, coords_grid, M,
    steps=15000,
    batch=1000,
    # Neuron-specific loss weights
    lambda_voxel=2.0,          # λ1: Weighted voxel
    lambda_aniso=0.1,          # λ2: Anisotropy
    lambda_sparse=0.0001,      # λ3: Sparsity (L1)
    lambda_tv=0.001,           # λ4: TV
    
    # RATE SCHEDULE (Beta)
    rate_warmup_steps=2000,    # Phase 1: 0 - 2000
    rate_prune_steps=5000,     # Phase 2: 2000 - 5000
    rate_beta_soft=1e-7,       # Phase 2 weight
    rate_beta_refine=1e-6,     # Phase 3 weight
    
    intensity_weight=10.0,
    aniso_target=0.5,          # RELAXED: 0.5 (was 0.1)
    tv_every=10,
    tv_patch_size=(8, 8, 8),
    
    # Densification
    densify_enabled=True,
    densify_from_iter=200,
    densify_until_iter=5000,
    max_gaussians=15000,       
    densify_every=250,
    grad_threshold=0.0004,     # RELAXED: 0.0004 (was 0.0006)
    min_amplitude=0.001,       # RELAXED: 0.001 (was 0.005)
    
    lr=3e-3,
    lr_final=5e-4
)

print(f"\nFinal model has {model.N} Gaussians")

Init Stats: Mean range: [-1.000, 1.000]
Model initialized with N=5000 Gaussians
NEURON-SPECIFIC TRAINING (w/ RD WARM-UP SCHEDULE)
L_Total = λ1*L_Voxel + λ2*L_Aniso + λ3*L_Sparse + λ4*L_TV + β(t)*L_Rate
  λ1 (voxel)     = 2.0000)
  λ2 (anisotropy)= 0.1000
  λ3 (sparsity)  = 0.0001
  λ4 (TV)        = 0.0010
  Rate Schedule (β):
    0 - 2000: 0.0 (Discovery)
    2000 - 5000: 1.0e-07 (Soft Pruning)
    5000+ : 1.0e-06 (Refinement)


Training:   1%|▏         | 205/15000 [00:07<08:55, 27.61it/s]

iter   200 | Vox=0.0825 Rate=0.0b (β=0.0e+00) Tot=0.1865 N=5000 | 7.4s


Training:   2%|▏         | 253/15000 [00:09<10:37, 23.14it/s]

  [Densify @250] N=6362 (+4 +1405 -47 -0)


...

iter 14600 | Vox=0.0029 Rate=51.9b (β=1.0e-06) Tot=0.0058 N=12235 | 913.0s


Training:  99%|█████████▊| 14802/15000 [15:25<00:12, 15.44it/s]

iter 14800 | Vox=0.0034 Rate=52.0b (β=1.0e-06) Tot=0.0069 N=12235 | 925.7s


Training: 100%|██████████| 15000/15000 [15:38<00:00, 15.98it/s]

iter 15000 | Vox=0.0034 Rate=52.0b (β=1.0e-06) Tot=0.0069 N=12235 | 938.4s

Final model has 12235 Gaussians


## 9) Full-volume reconstruction

In [11]:
# ============================================================================
# Forward Splatting Reconstruction: "Splat-to-Voxel"
# ============================================================================
# Instead of asking "Which Gaussians affect this voxel?" (O(voxels × N)),
# we ask "Which voxels does this Gaussian cover?" (O(N × footprint_size)).
#
# Strategy: Bounding Box Rasterization
#   1. Compute Covariance Matrix (Σ = R @ diag(s²) @ R^T) for each Gaussian
#   2. Determine 3σ Axis-Aligned Bounding Box from Σ
#   3. Local Summation: Only iterate through voxels within that specific box

@torch.no_grad()
def compute_covariance_and_bbox(mu, log_s, q, shape_zyx, n_sigma=3.0):
    """
    Compute covariance matrix and axis-aligned bounding box for each Gaussian.
    
    For anisotropic Gaussian with rotation R and scales s:
        Σ = R @ diag(s²) @ R^T
    
    The 3σ extent along axis i is: sqrt(Σ[i,i]) * n_sigma
    """
    Z, Y, X = shape_zyx
    device = mu.device
    N = mu.shape[0]
    
    # Scales and rotation
    s = torch.exp(log_s).clamp(1e-4, 10.0)  # (N, 3)
    qn = safe_normalize(q)
    R = quat_to_rotmat(qn)  # (N, 3, 3)
    
    # ===================================================================
    # Step 1: Compute Covariance Matrix Σ = R @ diag(s²) @ R^T
    # ===================================================================
    s_sq = s ** 2  # (N, 3) variance along principal axes
    # Σ = R @ diag(s²) @ R^T
    # Σ[i,j] = sum_k R[i,k] * s²[k] * R[j,k]
    # Using: RS = R * s (broadcasting), then Σ = RS @ RS^T
    RS = R * s[:, None, :]  # (N, 3, 3) * (N, 1, 3) -> (N, 3, 3)
    Sigma = torch.bmm(RS, RS.transpose(1, 2))  # (N, 3, 3) covariance matrix
    
    # ===================================================================
    # Step 2: Axis-Aligned Bounding Box from Covariance
    # ===================================================================
    # The 3σ extent along axis i is: n_sigma * sqrt(Σ[i,i])
    # Extract diagonal elements (variance along x, y, z)
    sigma_diag = torch.diagonal(Sigma, dim1=-2, dim2=-1)  # (N, 3) = [Σ_xx, Σ_yy, Σ_zz]
    std_xyz = sigma_diag.sqrt()  # Standard deviation along each axis (N, 3)
    
    # Convert mu from normalized [-1,1] to voxel coordinates
    mu_voxel = torch.zeros_like(mu)
    mu_voxel[:, 0] = (mu[:, 0] + 1) / 2 * (X - 1)  # x
    mu_voxel[:, 1] = (mu[:, 1] + 1) / 2 * (Y - 1)  # y
    mu_voxel[:, 2] = (mu[:, 2] + 1) / 2 * (Z - 1)  # z
    
    # Scale std to voxel units (normalized coords span [-1,1] -> [0, dim-1])
    std_voxel = std_xyz.clone()
    std_voxel[:, 0] = std_xyz[:, 0] * (X - 1) / 2
    std_voxel[:, 1] = std_xyz[:, 1] * (Y - 1) / 2
    std_voxel[:, 2] = std_xyz[:, 2] * (Z - 1) / 2
    
    # 3σ bounding box
    half_extent = std_voxel * n_sigma  # (N, 3)
    
    # Compute integer bounding box
    bbox_min = (mu_voxel - half_extent).floor().long()
    bbox_max = (mu_voxel + half_extent).ceil().long()
    
    # Clamp to volume bounds
    bbox_min[:, 0].clamp_(0, X - 1)
    bbox_min[:, 1].clamp_(0, Y - 1)
    bbox_min[:, 2].clamp_(0, Z - 1)
    bbox_max[:, 0].clamp_(0, X - 1)
    bbox_max[:, 1].clamp_(0, Y - 1)
    bbox_max[:, 2].clamp_(0, Z - 1)
    
    return s, R, mu_voxel, bbox_min, bbox_max, Sigma


@torch.no_grad()
def reconstruct_forward_splat(model: GaussianMixtureVolume, 
                               shape_zyx: Tuple[int,int,int],
                               spacing_xyz: Tuple[float,float,float],
                               n_sigma: float = 3.0) -> torch.Tensor:
    """
    Forward Splatting Reconstruction using Bounding Box Rasterization.
    
    For every Gaussian:
        1. Compute Covariance Matrix (Σ) - defines shape and orientation
        2. Determine 3σ Bounding Box - axis-aligned limits (x_min, x_max, ...)
        3. Local Summation - only iterate voxels within that specific box
    
    Args:
        model: GaussianMixtureVolume with mu, log_s, q, a, b parameters
        shape_zyx: Output volume shape (Z, Y, X)
        spacing_xyz: Voxel spacing (dx, dy, dz) - for reference
        n_sigma: Number of standard deviations for cutoff (default 3.0)
    
    Returns:
        Reconstructed volume (Z, Y, X)
    """
    Z, Y, X = shape_zyx
    device = model.mu.device
    
    # Initialize output volume (accumulator)
    volume = torch.zeros((Z, Y, X), device=device, dtype=torch.float32)
    
    # Get Gaussian parameters
    mu = model.mu.data          # (N, 3) in normalized coords [-1, 1]
    log_s = model.log_s.data    # (N, 3)
    q = model.q.data            # (N, 4)
    a = model.a.data            # (N,) amplitude
    b = model.b.data            # scalar bias
    
    # Compute covariance and bounding boxes
    s, R, mu_voxel, bbox_min, bbox_max, Sigma = compute_covariance_and_bbox(
        mu, log_s, q, shape_zyx, n_sigma
    )
    
    # ===================================================================
    # Step 3: Local Summation - Splat each Gaussian to its bounding box
    # ===================================================================
    N = model.N
    for i in tqdm(range(N), desc="Splatting Gaussians", leave=False):
        x0, y0, z0 = bbox_min[i].tolist()
        x1, y1, z1 = bbox_max[i].tolist()
        
        # Skip empty bounding boxes
        if x1 < x0 or y1 < y0 or z1 < z0:
            continue
        
        # Create local voxel grid for this Gaussian's footprint
        local_x = torch.arange(x0, x1 + 1, device=device, dtype=torch.float32)
        local_y = torch.arange(y0, y1 + 1, device=device, dtype=torch.float32)
        local_z = torch.arange(z0, z1 + 1, device=device, dtype=torch.float32)
        
        # Convert voxel coords to normalized coords for Gaussian evaluation
        local_x_norm = local_x / (X - 1) * 2 - 1
        local_y_norm = local_y / (Y - 1) * 2 - 1
        local_z_norm = local_z / (Z - 1) * 2 - 1
        
        # Create 3D meshgrid
        zz, yy, xx = torch.meshgrid(local_z_norm, local_y_norm, local_x_norm, indexing='ij')
        local_coords = torch.stack([xx, yy, zz], dim=-1)  # (lz, ly, lx, 3)
        local_shape = local_coords.shape[:3]
        local_coords_flat = local_coords.reshape(-1, 3)  # (P, 3)
        
        # Evaluate Gaussian at local voxels
        # Δx = coord - μ_i
        dx = local_coords_flat - mu[i:i+1]  # (P, 3)
        
        # Transform to Gaussian's principal axes: y = R^T @ Δx / s
        Rt_i = R[i].T      # (3, 3) inverse rotation
        s_i = s[i]         # (3,) scales
        y = (dx @ Rt_i) / (s_i + 1e-8)  # (P, 3)
        
        # Gaussian: g(x) = exp(-0.5 * ||y||²)
        mahal_sq = (y ** 2).sum(dim=-1)  # (P,) Mahalanobis distance squared
        g = torch.exp(-0.5 * mahal_sq)   # (P,)
        
        # Contribution: a_i * g(x)
        contrib = a[i] * g
        
        # Accumulate to volume
        volume[z0:z1+1, y0:y1+1, x0:x1+1] += contrib.reshape(local_shape)
    
    # Add global bias and clamp
    volume = volume + b
    return volume.clamp(0, 1)


@torch.no_grad()
def reconstruct_forward_splat_vectorized(model: GaussianMixtureVolume, 
                                          shape_zyx: Tuple[int,int,int],
                                          spacing_xyz: Tuple[float,float,float],
                                          n_sigma: float = 3.0,
                                          max_footprint_voxels: int = 50000) -> torch.Tensor:
    """
    Vectorized forward splatting with footprint size sorting.
    Processes Gaussians from smallest to largest footprint for memory efficiency.
    
    Args:
        max_footprint_voxels: Skip Gaussians with footprint larger than this
                              (they will be computed separately)
    """
    Z, Y, X = shape_zyx
    device = model.mu.device
    
    volume = torch.zeros((Z, Y, X), device=device, dtype=torch.float32)
    
    mu = model.mu.data
    log_s = model.log_s.data
    q = model.q.data
    a = model.a.data
    b = model.b.data
    
    s, R, mu_voxel, bbox_min, bbox_max, Sigma = compute_covariance_and_bbox(
        mu, log_s, q, shape_zyx, n_sigma
    )
    
    # Compute footprint size for each Gaussian
    bbox_size = bbox_max - bbox_min + 1  # (N, 3)
    footprint_voxels = bbox_size[:, 0] * bbox_size[:, 1] * bbox_size[:, 2]  # (N,)
    
    # Sort by footprint size (smallest first for memory efficiency)
    sorted_idx = footprint_voxels.argsort()
    
    N = model.N
    for idx in tqdm(range(N), desc="Splatting (sorted)", leave=False):
        i = sorted_idx[idx].item()
        
        x0, y0, z0 = bbox_min[i].tolist()
        x1, y1, z1 = bbox_max[i].tolist()
        
        if x1 < x0 or y1 < y0 or z1 < z0:
            continue
            
        footprint = (x1 - x0 + 1) * (y1 - y0 + 1) * (z1 - z0 + 1)
        if footprint > max_footprint_voxels:
            # For very large Gaussians, we could use a different strategy
            # For now, we still process them but warn
            pass
        
        # Create local grid
        local_x = torch.arange(x0, x1 + 1, device=device, dtype=torch.float32)
        local_y = torch.arange(y0, y1 + 1, device=device, dtype=torch.float32)
        local_z = torch.arange(z0, z1 + 1, device=device, dtype=torch.float32)
        
        local_x_norm = local_x / (X - 1) * 2 - 1
        local_y_norm = local_y / (Y - 1) * 2 - 1
        local_z_norm = local_z / (Z - 1) * 2 - 1
        
        zz, yy, xx = torch.meshgrid(local_z_norm, local_y_norm, local_x_norm, indexing='ij')
        local_coords = torch.stack([xx, yy, zz], dim=-1)
        local_shape = local_coords.shape[:3]
        local_coords_flat = local_coords.reshape(-1, 3)
        
        dx = local_coords_flat - mu[i:i+1]
        y = (dx @ R[i].T) / (s[i] + 1e-8)
        g = torch.exp(-0.5 * (y ** 2).sum(dim=-1))
        
        volume[z0:z1+1, y0:y1+1, x0:x1+1] += (a[i] * g).reshape(local_shape)
    
    volume = volume + b
    return volume.clamp(0, 1)


# Legacy method (backward query: evaluate all N Gaussians at each voxel)
@torch.no_grad()
def reconstruct_full(model: GaussianMixtureVolume, coords_grid: torch.Tensor, chunk=8_000):
    """
    Legacy backward reconstruction: O(voxels × N).
    Use forward splatting for large N.
    """
    Z, Y, X, _ = coords_grid.shape
    coords = coords_grid.reshape(-1, 3)
    out = torch.empty(coords.shape[0], device=coords.device, dtype=torch.float32)
    for s in tqdm(range(0, coords.shape[0], chunk), desc="Reconstruct (legacy)"):
        out[s:s+chunk] = model(coords[s:s+chunk])
    return out.reshape(Z, Y, X).clamp(0, 1)


# ============================================================================
# Run Forward Splatting Reconstruction
# ============================================================================
print("=" * 60)
print("Forward Splatting Reconstruction (Splat-to-Voxel)")
print("=" * 60)
print(f"Volume shape: {V_t.shape}")
print(f"Number of Gaussians: {model.N}")
print(f"Using n_sigma=3.0 for bounding box cutoff")
print()

V_hat = reconstruct_forward_splat(model, V_t.shape, VOXEL_SPACING, n_sigma=3.0)

print()
print(f"Reconstructed: {V_hat.shape}")
print(f"Value range: [{float(V_hat.min()):.4f}, {float(V_hat.max()):.4f}]")

Forward Splatting Reconstruction (Splat-to-Voxel)
Volume shape: torch.Size([100, 647, 813])
Number of Gaussians: 12235
Using n_sigma=3.0 for bounding box cutoff




Reconstructed: torch.Size([100, 647, 813])
Value range: [0.0000, 1.0000]


## 10) Metrics: PSNR/SSIM + neurite-map consistency (proxy)

In [12]:

@torch.no_grad()
def psnr(a: torch.Tensor, b: torch.Tensor, eps=1e-8):
    mse = ((a-b)**2).mean().clamp_min(eps)
    return float(10.0 * torch.log10(1.0 / mse).cpu())

@torch.no_grad()
def patch_ssim3d(a: torch.Tensor, b: torch.Tensor, win=7, K1=0.01, K2=0.03):
    a_ = a[None,None]
    b_ = b[None,None]
    pad = win//2
    mu_a = F.avg_pool3d(a_, win, stride=1, padding=pad)
    mu_b = F.avg_pool3d(b_, win, stride=1, padding=pad)
    sigma_a = F.avg_pool3d(a_*a_, win, stride=1, padding=pad) - mu_a*mu_a
    sigma_b = F.avg_pool3d(b_*b_, win, stride=1, padding=pad) - mu_b*mu_b
    sigma_ab= F.avg_pool3d(a_*b_, win, stride=1, padding=pad) - mu_a*mu_b

    C1 = (K1**2)
    C2 = (K2**2)
    ssim_map = ((2*mu_a*mu_b + C1) * (2*sigma_ab + C2)) / ((mu_a*mu_a + mu_b*mu_b + C1) * (sigma_a + sigma_b + C2) + 1e-8)
    return float(ssim_map.mean().cpu())

PSNR = psnr(V_t, V_hat)
SSIM = patch_ssim3d(V_t, V_hat, win=7)
print("PSNR:", PSNR, "SSIM:", SSIM)

M_hat = make_neurite_map(V_hat)
m_consistency = float((M_hat - M).abs().mean().cpu())
print("Neurite-map L1 (proxy):", m_consistency)


PSNR: 26.207958221435547 SSIM: 0.8296918869018555
Neurite-map L1 (proxy): 0.02582165226340294


## 11) Bitstream packing (baseline)
Quantize parameters and store as npz+gzip (starter).

In [13]:

import gzip, io

@torch.no_grad()
def quantize_params(model: GaussianMixtureVolume) -> Dict[str, np.ndarray]:
    mu   = (model.mu / Q.mu).round().to(torch.int32).cpu().numpy()
    logs = (model.log_s / Q.log_s).round().to(torch.int16).cpu().numpy()
    q    = (safe_normalize(model.q) / Q.q).round().to(torch.int16).cpu().numpy()
    a    = (model.a / Q.a).round().to(torch.int16).cpu().numpy()
    b    = (model.b / Q.b).round().to(torch.int16).cpu().numpy()
    return {"mu": mu, "log_s": logs, "q": q, "a": a, "b": b,
            "Q_mu": np.array([Q.mu], np.float32),
            "Q_log_s": np.array([Q.log_s], np.float32),
            "Q_q": np.array([Q.q], np.float32),
            "Q_a": np.array([Q.a], np.float32),
            "Q_b": np.array([Q.b], np.float32)
           }

def save_bitstream_npz_gz(params: Dict[str,np.ndarray], out_path: str):
    bio = io.BytesIO()
    np.savez_compressed(bio, **params)
    raw = bio.getvalue()
    with gzip.open(out_path, "wb", compresslevel=9) as f:
        f.write(raw)
    return len(raw), os.path.getsize(out_path)

OUT_BITSTREAM = "neurogs_codec_stream.npz.gz"
raw_sz, gz_sz = save_bitstream_npz_gz(quantize_params(model), OUT_BITSTREAM)
print("Saved:", OUT_BITSTREAM)
print("Raw npz bytes:", raw_sz, "Gzip bytes:", gz_sz)

bpp = (gz_sz * 8) / (V_t.numel())
print("Bits per voxel (bpp):", bpp)


Saved: neurogs_codec_stream.npz.gz
Raw npz bytes: 220354 Gzip bytes: 219199
Bits per voxel (bpp): 0.03333755377739249


## 12) Decode + reconstruct from bitstream (verification)

In [ ]:

@torch.no_grad()
def load_bitstream_npz_gz(path: str) -> Dict[str, np.ndarray]:
    with gzip.open(path, "rb") as f:
        raw = f.read()
    bio = io.BytesIO(raw)
    data = np.load(bio)
    return {k: data[k] for k in data.files}

@torch.no_grad()
def build_model_from_quantized(qparams: Dict[str,np.ndarray], device: str) -> GaussianMixtureVolume:
    Qmu = float(qparams["Q_mu"][0])
    Qls = float(qparams["Q_log_s"][0])
    Qq  = float(qparams["Q_q"][0])
    Qa  = float(qparams["Q_a"][0])
    Qb  = float(qparams["Q_b"][0])

    mu = torch.from_numpy(qparams["mu"]).to(device).float() * Qmu
    log_s = torch.from_numpy(qparams["log_s"]).to(device).float() * Qls
    q = torch.from_numpy(qparams["q"]).to(device).float() * Qq
    a = torch.from_numpy(qparams["a"]).to(device).float() * Qa
    b = torch.from_numpy(qparams["b"]).to(device).float().view(()) * Qb

    m = GaussianMixtureVolume(mu.shape[0], mu, a).to(device)
    m.log_s.data.copy_(log_s)
    m.q.data.copy_(q)
    m.b.data.copy_(b)
    return m

qparams = load_bitstream_npz_gz(OUT_BITSTREAM)
model_dec = build_model_from_quantized(qparams, DEVICE)

V_hat2 = reconstruct_full(model_dec, coords_grid)
print("Decode PSNR:", psnr(V_t, V_hat2), "SSIM:", patch_ssim3d(V_t, V_hat2, win=7))
print("Max abs diff between V_hat and V_hat2:", float((V_hat - V_hat2).abs().max().cpu()))


Reconstruct (legacy):   0%|          | 17/6576 [00:00<00:42, 155.23it/s]

## 13) Densify / prune hooks (optional)

In [ ]:
# ============================================================================
# Densification Analysis (optional)
# ============================================================================
# The DensificationController and train() with densification are now integrated 
# in the training cell above. This cell is for post-training analysis.
import matplotlib.pyplot as plt
if 'densify_log' in dir() and densify_log:
    print("Densification history:")
    for it, stats in densify_log:
        print(f"  iter {it+1}: N={stats['total_gaussians']} "
              f"(+{stats['cloned']} clone, +{stats['split']} split, "
              f"-{stats['pruned_amp']} amp, -{stats['pruned_scale']} scale)")
    
    # Plot Gaussian count over training
    if 'losses' in dir() and "N" in losses:
        plt.figure(figsize=(10, 4))
        plt.plot(losses["N"])
        plt.xlabel("Iteration")
        plt.ylabel("Number of Gaussians")
        plt.title("Gaussian Count During Training")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
else:
    print("No densification log available. Run training with densify_enabled=True first.")

## 14) Save checkpoint

In [ ]:
# # Clear GPU memory before training
# gc.collect()
# torch.cuda.empty_cache()

# # Reinitialize model with fresh state
# init_means, init_amp = init_gaussians_from_neurite_map(coords_grid, V_t, M, N0)
# model = GaussianMixtureVolume(N0, init_means, init_amp).to(DEVICE)
# print("Model N:", model.N)

# # Train with smaller topology patch to avoid OOM
# losses, densify_log = train(
#     model, V_t, coords_grid, M,
#     steps=3000,
#     batch=2_000,
#     topo_patch=(8, 16, 16),  # Reduced from (16,32,32) to avoid OOM
#     topo_every=50,           # Less frequent topology loss
#     densify_enabled=True,
#     max_gaussians=8000       # Limit max Gaussians
# )

In [ ]:

CKPT_PATH = "neurogs_codec_ckpt_with reg_with_dens_3.pt"

# Save full checkpoint with model, entropy models, and training config
config = {
    # Model parameters
    "N0": N0,
    "model_N": model.N,
    
    # Training hyperparameters
    "steps": 3000,
    "batch": 2_000,
    "lr": 5e-3,
    "kappa": 4.0,
    "lam": 0.001,
    "alpha": 0.01,
    "beta_sparse": 0.01,
    "beta_smooth": 0.005,
    "beta_overlap": 0.01,
    "topo_every": 20,
    "topo_patch": (8, 16, 16),
    
    # Densification parameters
    "densify_enabled": True,
    "densify_from_iter": 500,
    "densify_until_iter": 2500,
    "densify_every": 200,
    "grad_threshold": 0.0005,
    "min_amplitude": 0.002,
    "max_gaussians": 8000,
    
    # Data info
    "tif_path": TIF_PATH,
    "voxel_spacing": VOXEL_SPACING,
    "shape_zyx": tuple(V_t.shape),
}

torch.save({
    "model_state": model.state_dict(),
    "entropy_state": {
        "H_mu": H_mu.state_dict(),
        "H_logs": H_logs.state_dict(),
        "H_q": H_q.state_dict(),
        "H_a": H_a.state_dict(),
        "H_b": H_b.state_dict(),
    },
    "Q": Q.__dict__,
    "config": config,
    "losses": losses if 'losses' in dir() else None,
    "densify_log": densify_log if 'densify_log' in dir() else None,
}, CKPT_PATH)

print("Saved checkpoint:", CKPT_PATH)
print("\nConfig saved:")
for k, v in config.items():
    print(f"  {k}: {v}")


In [ ]:

# show the mip projection of the reconstructed volume vs original
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original MIP
mip_orig = V_t.max(dim=0).values.cpu().numpy()
axes[0].imshow(mip_orig, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Original MIP XY')
axes[0].axis('off')

# Reconstructed MIP (auto-scaled for visibility)
mip_recon = V_hat.max(dim=0).values.cpu().numpy()
axes[1].imshow(mip_recon, cmap='gray')  # auto-scale
axes[1].set_title(f'Reconstructed MIP XY\nmin={mip_recon.min():.3f}, max={mip_recon.max():.3f}')
axes[1].axis('off')

# Reconstructed MIP normalized to [0,1] for comparison
mip_recon_norm = (mip_recon - mip_recon.min()) / (mip_recon.max() - mip_recon.min() + 1e-8)
axes[2].imshow(mip_recon_norm, cmap='gray', vmin=0, vmax=1)
axes[2].set_title('Reconstructed MIP (normalized)')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"V_hat stats: min={float(V_hat.min()):.4f}, max={float(V_hat.max()):.4f}, mean={float(V_hat.mean()):.4f}")
print(f"V_t stats:   min={float(V_t.min()):.4f}, max={float(V_t.max()):.4f}, mean={float(V_t.mean()):.4f}")


In [ ]:
# diff map
diff_map = (V_hat - V_t).abs().cpu().numpy()
plt.figure(figsize=(6,6))
plt.imshow(np.max(diff_map, axis=0), cmap='hot')
plt.title('Absolute Difference MIP XY')
plt.axis('off')
plt.colorbar()
plt.show()
def patch_ssim_loss(model: GaussianMixtureVolume, coords_grid: torch.Tensor,
                    V: torch.Tensor, patch_zyx=(8,16,16)) -> torch.Tensor:
    """
    SSIM loss on random patches to encourage structural similarity.
    """
    (z0,y0,x0), tgt_patch = extract_random_patch(V, patch_zyx)
    pz,py,px = patch_zyx
    coords = coords_grid[z0:z0+pz, y0:y0+py, x0:x0+px].reshape(-1,3)
    pred_patch = model(coords).reshape(pz,py,px)
    
    # SSIM loss (1 - SSIM)
    ssim_val = patch_ssim3d(pred_patch, tgt_patch, win=7)
    return torch.tensor(1.0 - ssim_val, device=V.device)